In [1]:
import time
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd
import sys

# -----------------------------
# Inputs (FULL FILE PATHS)
# -----------------------------
shapefile_path = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Alaska\Alaska_3_10.shp"
ghi_raster = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Alaska\Alaska_GHI_Mosaic_3338.tif"
output_csv = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Alaska\Alaska_GHI.csv"
log_file = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Alaska\Alaska_log.txt"

# -----------------------------
# Logger (FLUSH ENABLED)
# -----------------------------
def log(msg):
    print(msg, flush=True)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Start Script
# -----------------------------
log("Starting Alaska GHI CSV export script.")

# -----------------------------
# Load Shapefile
# -----------------------------
log("Loading shapefile...")
gdf = gpd.read_file(shapefile_path)
total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# Check CRS
# -----------------------------
log("Checking CRS information...")

vector_crs = gdf.crs
log(f"Shapefile CRS: {vector_crs}")

with rasterio.open(ghi_raster) as src:
    raster_crs = src.crs

log(f"Raster CRS: {raster_crs}")

if vector_crs != raster_crs:
    log("WARNING: CRS mismatch detected!")
    log("Reproject shapefile to match raster before running for accurate results.")
else:
    log("CRS match confirmed.")

# -----------------------------
# Compute Zonal Statistics with Progress
# -----------------------------
log("Beginning zonal statistics computation...")

start_time = time.time()
ghi_means = []

for idx, row in gdf.iterrows():
    
    stat = zonal_stats(
        row.geometry,
        ghi_raster,
        stats=["mean"],
        all_touched=True,
        nodata=-9999
    )[0]["mean"]
    
    ghi_means.append(stat)

    # ---- Progress Update ----
    percent_complete = ((idx + 1) / total_polygons) * 100

    # Print every 1% or final row
    if (idx + 1) % max(1, total_polygons // 100) == 0 or (idx + 1) == total_polygons:
        elapsed = time.time() - start_time
        log(f"Progress: {percent_complete:.1f}% | "
            f"{idx+1}/{total_polygons} polygons | "
            f"Elapsed: {elapsed/60:.2f} minutes")

# -----------------------------
# Build Output DataFrame
# -----------------------------
log("Building output dataframe...")

output_df = pd.DataFrame({
    "ROW_ID": gdf["ROW_ID"],
    "GHI_Mean": ghi_means
})

# -----------------------------
# Export CSV
# -----------------------------
output_df.to_csv(output_csv, index=False)
log(f"CSV successfully saved: {output_csv}")

total_time = (time.time() - start_time) / 60
log(f"Script completed successfully in {total_time:.2f} minutes.")

Starting Alaska GHI CSV export script.
Loading shapefile...
Loaded 2703 polygons.
Checking CRS information...
Shapefile CRS: EPSG:3338
Raster CRS: EPSG:3338
CRS match confirmed.
Beginning zonal statistics computation...
Progress: 1.0% | 27/2703 polygons | Elapsed: 0.00 minutes
Progress: 2.0% | 54/2703 polygons | Elapsed: 0.01 minutes
Progress: 3.0% | 81/2703 polygons | Elapsed: 0.01 minutes
Progress: 4.0% | 108/2703 polygons | Elapsed: 0.01 minutes
Progress: 5.0% | 135/2703 polygons | Elapsed: 0.02 minutes
Progress: 6.0% | 162/2703 polygons | Elapsed: 0.02 minutes
Progress: 7.0% | 189/2703 polygons | Elapsed: 0.02 minutes
Progress: 8.0% | 216/2703 polygons | Elapsed: 0.02 minutes
Progress: 9.0% | 243/2703 polygons | Elapsed: 0.03 minutes
Progress: 10.0% | 270/2703 polygons | Elapsed: 0.03 minutes
Progress: 11.0% | 297/2703 polygons | Elapsed: 0.03 minutes
Progress: 12.0% | 324/2703 polygons | Elapsed: 0.03 minutes
Progress: 13.0% | 351/2703 polygons | Elapsed: 0.04 minutes
Progress: 14